In [ ]:
!pip install requests beautifulsoup4 pandas -q

In [4]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from getpass import getpass
from datetime import datetime
import time


In [ ]:
def fetch_football_news(page_size=10, language='en'):
    url = 'https://newsapi.org/v2/everything'

    query = '("Premier League" OR "La Liga" OR "Champions League" OR "Europa League" OR "Serie A" OR "Bundesliga" OR "football")'

    params = {
        'q': query,
        'language': language,
        'sortBy': 'publishedAt',
        'pageSize': page_size,
        'apiKey': NEWS_API_KEY,
    }

    res = requests.get(url, params=params, timeout=15)
    res.raise_for_status()

    data = res.json()

    if data.get('status') != 'ok':
        raise Exception(data)

    return data.get('articles', [])

def get_full_article(url):
  headers = {
      'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0 Safari/537.36'
  }

  res = requests.get(url, headers=headers, timeout=15)
  res.raise_for_status()

  soup = BeautifulSoup(res.text, 'html.parser')

  for tag in soup(['script', 'style', 'nav', 'footer', 'header', 'aside', 'form', 'button']):
      tag.decompose()

  paragraphs = soup.find_all('p')

  texts = []
  for p in paragraphs:
      text = p.get_text(' ', strip=True)
      if len(text) >= 40:
          texts.append(text)

  full_text = '\n'.join(texts)
  return full_text


In [7]:
import json

LEAGUE_TAGS = {
    'Premier League': ['premier-league', 'epl'],
    'Champions League': ['champions-league', 'ucl'],
    'Europa League': ['europa-league', 'uel'],
    'La Liga': ['la-liga'],
    'Serie A': ['serie-a'],
    'Bundesliga': ['bundesliga'],
}

def extract_tags(text):
    tags = set()
    lower_text = text.lower()

    for league, league_tags in LEAGUE_TAGS.items():
        if league.lower() in lower_text:
            tags.update(league_tags)

    return sorted(tags)


articles = fetch_football_news(page_size=10, language='en')

rows = []

for i, article in enumerate(articles, start=1):
    title = article.get('title')
    description = article.get('description')
    newsapi_content = article.get('content')
    source = article.get('source', {}).get('name')
    published_at = article.get('publishedAt')
    url = article.get('url')

    print(f'[{i}] {title}')

    try:
        full_text = get_full_article(url)
        time.sleep(1)
    except Exception as e:
        full_text = ''
        print('본문 수집 실패:', e)

    combined_text = ' '.join([
        title or '',
        description or '',
        newsapi_content or '',
        full_text or '',
    ])

    tags = extract_tags(combined_text)

    rows.append({
        'title': title,
        'source': source,
        'publishedAt': published_at,
        'description': description,
        'newsapi_content': newsapi_content,
        'full_text': full_text,
        'url': url,
        'tags': tags,
    })

df = pd.DataFrame(rows)

json_result = df.to_json(
    orient='records',
    force_ascii=False,
    indent=2
)

print(json_result)

[1] Borussia Dortmund vs Eintracht Frankfurt – Predicted lineup
본문 수집 실패: 401 Client Error: Unauthorized for url: https://consent.yahoo.com/v2/collectConsent?sessionId=1_cc-session_e1e0ca2e-1464-400e-ad69-961d3bef4195
[2] Borussia Dortmund vs Frankfurt – Match preview and team news
[3] Free Long Island soccer academy ‘removing all socioeconomic barriers’ to develop next generation
[4] Colbert Mocks Marco Rubio’s ‘Small Crystal Football’ Gift to Pope Leo: ‘They Got It With Their Sports Illustrated’ | Video
[5] Alexander-Arnold omission 'mind-boggling' to Rooney
[6] Aurelien Tchouameni: Man United hold talks as situation in Madrid explodes
[7] Copa Libertadores game in Colombia abandoned due to crowd trouble
[8] How Pittsburgh Insurance Agent Ended Up on Stage at 2026 NFL Draft
[9] Increasing flat racing’s appeal in Ireland is still a conundrum
[
  {
    "title":"Borussia Dortmund vs Eintracht Frankfurt – Predicted lineup",
    "source":"Yahoo Entertainment",
    "publishedAt":"2026-05-0